# Notebook 01 — Preprocessing Evaluation

Evaluates Module 1 (Preprocessor):
1. Artificially degrade 20 clean IAM images (rotate, add noise, reduce contrast)
2. Run TrOCR on raw vs preprocessed versions
3. Report before/after CER delta per step
4. Generate before/after image gallery

**Self-contained**: This notebook loads IAM data directly. No dependency on notebook 00.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import cv2
import random
import numpy as np
import matplotlib.pyplot as plt
import jiwer
from PIL import Image
from pathlib import Path

from src.preprocessing import Preprocessor
from src.ocr import TrOCREngine

print('Imports OK')

In [ ]:
# ── IAM words.txt loader (self-contained) ─────────────────────────────────────
# Requires manual registration + download from:
# https://www.fki.inf.unibe.ch/databases/iam-handwriting-database
# Place files under ../data/iam/

IAM_DIR = Path('../data/iam')
assert IAM_DIR.exists(), f'IAM data not found at {IAM_DIR}. Download it first.'

words_file = IAM_DIR / 'ascii' / 'words.txt'
samples = []
with open(words_file) as f:
    for line in f:
        if line.startswith('#'): continue
        parts = line.strip().split(' ')
        if len(parts) < 9: continue
        word_id = parts[0]
        label = parts[-1]
        parts_id = word_id.split('-')
        img_path = IAM_DIR / 'words' / parts_id[0] / f'{parts_id[0]}-{parts_id[1]}' / f'{word_id}.png'
        if img_path.exists():
            samples.append({'id': word_id, 'label': label, 'path': str(img_path)})

print(f'Total IAM word samples found: {len(samples)}')

In [ ]:
# ── Degradation Functions ─────────────────────────────────────────────────────

def add_rotation(img_arr, angle=15):
    h, w = img_arr.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    return cv2.warpAffine(img_arr, M, (w, h), borderValue=255)

def add_noise(img_arr, sigma=25):
    noise = np.random.normal(0, sigma, img_arr.shape).astype(np.int16)
    return np.clip(img_arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)

def reduce_contrast(img_arr, factor=0.4):
    mid = 128
    return np.clip(mid + (img_arr.astype(np.float32) - mid) * factor, 0, 255).astype(np.uint8)

def combined_degradation(img_arr):
    out = add_rotation(img_arr, 12)
    out = add_noise(out, 30)
    out = reduce_contrast(out, 0.5)
    return out

print('Degradation functions defined.')

In [ ]:
# ── Degrade 20 random IAM samples ────────────────────────────────────────────
subset = random.sample(samples, min(20, len(samples)))
results = []

engine = TrOCREngine()

for s in subset:
    img = Image.open(s['path']).convert('RGB')
    arr = np.array(img)
    degraded = combined_degradation(arr)
    results.append({
        'label': s['label'],
        'original': img,
        'degraded': Image.fromarray(degraded),
    })

print(f'Prepared {len(results)} degraded samples.')

In [ ]:
# ── Evaluate: raw degraded vs preprocessed (per-step ablation) ───────────────
configs = {
    'Raw (degraded)':     {'deskew': False, 'denoise': False, 'contrast': False, 'binarize': False},
    'Deskew only':        {'deskew': True,  'denoise': False, 'contrast': False, 'binarize': False},
    'Denoise only':       {'deskew': False, 'denoise': True,  'contrast': False, 'binarize': False},
    'CLAHE only':         {'deskew': False, 'denoise': False, 'contrast': True,  'binarize': False},
    'Sauvola only':       {'deskew': False, 'denoise': False, 'contrast': False, 'binarize': True},
    'Full pipeline':      {'deskew': True,  'denoise': True,  'contrast': True,  'binarize': True},
}

cer_results = {}
for name, config in configs.items():
    pre = Preprocessor(config=config)
    preds, refs = [], []
    for r in results:
        cleaned = pre.transform(r['degraded'])
        candidates = engine.recognise(cleaned)
        preds.append(candidates[0].word if candidates else '')
        refs.append(r['label'])
    cer = jiwer.cer(refs, preds)
    cer_results[name] = cer
    print(f'{name:30s} CER: {cer*100:.1f}%')

print('\nDone. Lower CER = better preprocessing for these degraded samples.')

In [ ]:
# ── Plot CER bar chart ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

names = list(cer_results.keys())
values = [v * 100 for v in cer_results.values()]

plt.figure(figsize=(10, 5))
bars = plt.bar(names, values, color=['#e17055' if n == 'Raw (degraded)' else '#6c5ce7' for n in names])
plt.xticks(rotation=20, ha='right')
plt.ylabel('CER (%)')
plt.title('Preprocessing Ablation — CER per Step')
for bar, v in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{v:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
os.makedirs('../outputs', exist_ok=True)
plt.savefig('../outputs/preprocessing_ablation_cer.png', dpi=100)
plt.show()
print('Saved to ../outputs/preprocessing_ablation_cer.png')